# Planning a two-target RNA assay
All measurements below are synthetic. Each complete request samples distinct sites and has a fixed molecular budget.

In [1]:
import pandas as pd
from design_atlas import policies, evaluate
from sampling_design import repeat_plan
frame = pd.read_csv('synthetic_pilot.csv').query("section == 'Synthetic-A' and spatial_cell >= 0").reset_index(drop=True)
records = []
for count in (6, 12, 20, 30):
    for method in ('random', 'systematic_grid', 'stratified_dispersed', 'maximin', 'inhibition'):
        ids, weights, valid = policies(frame, count, 200, 42 + count, method)
        row = evaluate(frame, ids, weights, valid, 600 // count, {'primary': ('Acan', 'Col2a1')})[0]
        records.append({'sites': count, 'method': method, 'recovery': row['dual_probability'], 'RMSE': row['rmse_per10k']})
result = pd.DataFrame(records)
print(result.round(3).to_string(index=False))

 sites               method  recovery   RMSE
     6               random     0.499 58.241
     6      systematic_grid     0.494 53.815
     6 stratified_dispersed     0.483 51.406
     6              maximin     0.521 53.792
     6           inhibition     0.491 55.607
    12               random     0.484 54.330
    12      systematic_grid     0.491 54.207
    12 stratified_dispersed     0.483 52.833
    12              maximin     0.516 55.118
    12           inhibition     0.473 53.311
    20               random     0.480 53.405
    20      systematic_grid     0.498 52.437
    20 stratified_dispersed     0.483 52.219
    20              maximin     0.484 51.821
    20           inhibition     0.477 52.353
    30               random     0.480 53.006
    30      systematic_grid     0.489 53.112
    30 stratified_dispersed     0.481 52.210
    30              maximin     0.496 52.821
    30           inhibition     0.489 53.036


In [2]:
at_twelve = result.query('sites == 12')
selected = at_twelve.loc[at_twelve.recovery.idxmax()]
print('Largest simulated recovery:', selected.method)
print('Lowest simulated error:', at_twelve.loc[at_twelve.RMSE.idxmin(), 'method'])
print('Independent repeats at 80% and 95%:', [repeat_plan(float(selected.recovery), t) for t in (.80, .95)])

Largest simulated recovery: maximin
Lowest simulated error: stratified_dispersed
Independent repeats at 80% and 95%: [3, 5]
